# Sensitivity analysis

1. **Optimal plane:** `run_pipeline(base)` sizes the plane for mission range + reserve and feeds the OpenVSP parasite drag back until it settles.
2. **One-at-a-time sweeps:** starting from that plane, one parameter is changed while everything else stays at its optimum, and the pipeline is run again.
3. **Two plots per sweep:**
   - **Range** with the fuel fixed at the optimal plane's fuel mass: how far does that tank get?
   - **Required fuel** [l] for mission range + reserve, with the 5–10 l target band shaded.

Cases where the mission can't be reached or the iteration doesn't converge are left out of the curves. Every case runs OpenVSP 2–3 times (~1.3 s each), so the sweeps take several minutes; lower `N_POINTS` for a quick look.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import ardea_sizer as az
from ardea_sizer.pipeline import evaluate_from_fuel_mass

FUEL_DENSITY = 0.75      # kg/l, gasoline
FUEL_BAND = (5.0, 10.0)  # l, target fuel band shaded in the required-fuel plot
N_POINTS = 9             # per sweep

## 1. Optimal plane from `base.json`

`base_opt` holds the converged OpenVSP drag (`c_d_0_wing`, `f_fixed`), so every sweep starts close to converged.

In [ ]:
base = az.load("base.json")

plane, base_opt = az.run_pipeline(base)

FUEL_OPT = float(plane.fuel_mass)                                    # kg, fixed for the range plot
RANGE_OPT = float(plane.range) / 1000                                # km
TARGET_KM = (base.mission.range + base.mission.reserve) / 1000       # km
CD0_OPT = float(plane.c_d_0)

print(f"Fuel:           {FUEL_OPT:8.3f} kg = {FUEL_OPT / FUEL_DENSITY:.2f} l")
print(f"Range:          {RANGE_OPT:8.1f} km  (mission + reserve {TARGET_KM:.0f} km)")
print(f"Wing area:      {float(plane.wing_area):8.3f} m²")
print(f"CD0:            {CD0_OPT:8.5f}")
print(f"Power required: {plane.power_req / 1000:8.2f} kW of {plane.power_available:.2f} kW")

## 2. Sweeps

`dotted path in base.json: (label, min, max)`.

- **$C_{D_0}$** is not an input of the pipeline: OpenVSP computes `c_d_0_wing` and `f_fixed` and would overwrite any value set here. So for this sweep the converged OpenVSP drag of `base_opt` is scaled by $C_{D_0} / C_{D_0,opt}$ and the plane is sized with `evaluate` alone (no OpenVSP feedback). The x-axis is $C_{D_0}$ at the optimal wing area.
- **$\alpha_{0L}$** only sets the cruise angle of attack (wing incidence), so it doesn't change fuel or range in the current model: its curves are flat.
- **$\alpha_{C_{L,max}}$** is not a parameter of the model (`base.json` has none): $C_{L,max}$ is given directly by `c_l_max_2d_takeoff`, so it is not swept.

In [ ]:
SWEEPS = {
    "mission.cruise_speed":           ("Cruise speed $v_{cruise}$ [m/s]",   60.0, 90.0),
    "mission.stall_speed":            ("Stall speed $v_{stall}$ [m/s]",     15.0, 26.0),
    "assumptions.wing_ar":            ("Aspect ratio $AR$",                 3.0,  12.0),
    "assumptions.oswald_eff":         ("Oswald efficiency $e$",             0.6,  0.95),
    "assumptions.delta_c_l_flap":     ("Flap $\\Delta C_L$",                0.0,  1.0),
    "assumptions.c_l_max_2d_takeoff": ("Airfoil $c_{l,max}$ (take-off)",    1.1,  1.9),
    "assumptions.alpha_0_l_cruise":   ("$\\alpha_{0L}$ [deg]",              -5.0, 0.0),
    "c_d_0":                          ("$C_{D_0}$",                         0.7 * CD0_OPT, 1.5 * CD0_OPT),
}


def get_value(b, path):
    if path == "c_d_0":
        return CD0_OPT
    obj = b
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj


def run_case(path, x, fuel_mass=None):
    # plane with `path` set to `x` (all else at base_opt), or None where the mission can't be reached
    try:
        if path == "c_d_0":
            k, a = x / CD0_OPT, base_opt.assumptions
            b = base_opt.with_(**{"assumptions.c_d_0_wing": k * a.c_d_0_wing, "assumptions.f_fixed": k * a.f_fixed})
            return az.evaluate(b) if fuel_mass is None else evaluate_from_fuel_mass(b, fuel_mass=fuel_mass)
        return az.run_pipeline(base_opt.with_(**{path: x}), fuel_mass=fuel_mass, verbose=False)[0]
    except (ValueError, RuntimeError):
        return None


def power_ok(p):
    return p is not None and p.power_req <= p.power_available * 1000


results = {}
for path, (label, lo, hi) in SWEEPS.items():
    values = np.linspace(lo, hi, N_POINTS)
    sized = [run_case(path, x) for x in values]                       # fuel for mission + reserve
    fixed = [run_case(path, x, fuel_mass=FUEL_OPT) for x in values]   # fuel fixed at FUEL_OPT
    results[path] = pd.DataFrame({
        "value": values,
        "fuel_l": [np.nan if p is None else float(p.fuel_mass) / FUEL_DENSITY for p in sized],
        "fuel_power_ok": [power_ok(p) for p in sized],
        "range_km": [np.nan if p is None else float(p.range) / 1000 for p in fixed],
        "range_power_ok": [power_ok(p) for p in fixed],
    })
    print(f"{label}: done")

In [ ]:
# chart style
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
SERIES = "#2a78d6"


def style(ax, label):
    ax.set_title(label, color=INK, fontsize=11, loc="left")
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(MaxNLocator(5))
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)


def plot_sweeps(column, ok_column, baseline, ylabel, title, band=None, ref=None):
    n_cols = 4
    n_rows = -(-len(SWEEPS) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.6 * n_rows), sharey=True, constrained_layout=True)

    for ax, (path, (label, *_)) in zip(axes.flat, SWEEPS.items()):
        df = results[path]
        if band is not None:
            ax.axhspan(*band, color=GRID, alpha=0.6, lw=0, zorder=0, label=f"{band[0]:g}–{band[1]:g} l")
        if ref is not None:
            ax.axhline(ref[0], color=INK_MUTED, lw=1.5, ls="--", zorder=1, label=ref[1])
        ax.plot(df.value, df[column], color=SERIES, lw=2, zorder=2)
        bad = df[~df[ok_column]]
        ax.plot(bad.value, bad[column], "o", ms=6, mfc="white", mec=SERIES, mew=1.5, zorder=3,
                label="$P_{req} > P_{max}$")
        ax.plot(get_value(base_opt, path), baseline, "o", ms=8, color=SERIES, mec="white", mew=2, zorder=4,
                label="optimal plane")
        style(ax, label)

    for ax in axes.flat[len(SWEEPS):]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel, color=INK_MUTED)
    axes.flat[0].legend(frameon=False, fontsize=8, labelcolor=INK_MUTED)
    fig.suptitle(title, color=INK, fontsize=13, x=0.01, ha="left")
    plt.show()

### Range with fixed fuel

Fuel held at the optimal plane's fuel mass. Dashed line = mission range + reserve: above it the tank is enough. Hollow points: cruise power required (at MTOW) exceeds $P_{max}$.

In [ ]:
plot_sweeps(
    "range_km", "range_power_ok", RANGE_OPT,
    ylabel="Range [km]",
    title=f"Range vs. parameters  (fuel fixed at {FUEL_OPT:.2f} kg = {FUEL_OPT / FUEL_DENSITY:.1f} l, all else at the optimum)",
    ref=(TARGET_KM, "mission + reserve"),
)

### Required fuel

Fuel sized for mission range + reserve, converted at `FUEL_DENSITY`. Grey band = 5–10 l target. Hollow points: cruise power required (at MTOW) exceeds $P_{max}$.

In [ ]:
plot_sweeps(
    "fuel_l", "fuel_power_ok", FUEL_OPT / FUEL_DENSITY,
    ylabel="Required fuel [l]",
    title=f"Required fuel vs. parameters  ({TARGET_KM:.0f} km incl. reserve, all else at the optimum)",
    band=FUEL_BAND,
)

In [ ]:
pd.DataFrame(
    {
        "optimum": [get_value(base_opt, p) for p in SWEEPS],
        "fuel at min [l]": [results[p].fuel_l.iloc[0] for p in SWEEPS],
        "fuel at max [l]": [results[p].fuel_l.iloc[-1] for p in SWEEPS],
        "range at min [km]": [results[p].range_km.iloc[0] for p in SWEEPS],
        "range at max [km]": [results[p].range_km.iloc[-1] for p in SWEEPS],
    },
    index=[label for label, *_ in SWEEPS.values()],
).round(3)